# 04 Resultaten inlezen en weergeven

In dit script worden de modelresultaten weergegeven en geplot.

In [19]:
import shutil
import logging
import numpy as np
import pandas as pd
import geopandas as gpd
import xarray as xr
import rioxarray
import hkvsobekpy
from pathlib import Path
import matplotlib.pyplot as plt
from shapely.geometry import Point

In [20]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


### Inlezen meetlocaties en meetdata

Functie voor inlezen csv met afvoermetingen of grondwaterstanden

In [21]:
def inlezen_csv_met_metadata(file_path: Path):
    meta = {}
    with open(file_path, "r", encoding="utf-8") as f:
        lines = f.readlines()

    for line in lines:
        if not line.startswith("#"):
            continue
        text = line.strip("#").strip()
        if ":" in text:
            key, value = text.split(":", 1)
            meta[key.strip()] = value.strip().strip("; ")

    header_idx = next(i for i, line in enumerate(lines) if "Tijdstip (UTC);Waarde" in line)

    df = pd.read_csv(
        file_path,
        sep=";",
        skiprows=header_idx + 1,
        header=None,
        names=["Tijdstip (UTC)", "Waarde"],
        decimal=",",
        skipinitialspace=True
    )

    df["Tijdstip (UTC)"] = pd.to_datetime(df["Tijdstip (UTC)"], utc=True)
    df["time"] = df["Tijdstip (UTC)"].dt.tz_convert("Europe/Amsterdam").dt.tz_localize(None)
    df["Waarde"] = df["Waarde"].replace("---", np.nan).str.replace(",", ".").astype(float)
    df = df.set_index("time")[["Waarde"]]
    df.columns = [csv_file.stem]
    x = float(meta["Postitie X"].split(";")[0].strip("; (RD)"))
    y = float(meta["Postitie Y"].split(";")[0].strip("; (RD)"))

    return meta, Point(x,y), df

INPUT vanuit WRIJ voor RR unpaved methode

In [22]:
main_dir = Path("D:\\153961_Oude_IJssel")
path_dir_data = Path(main_dir, "WRIJ_RR_Unpaved_methode_01_data")
path_oplevering_amigo = Path(path_dir_data, "20260508_AMIGOtifs_output")
path_overig = Path(path_oplevering_amigo, "04_Overig")
path_maaiveld = Path(path_overig, "MV25_FILL.ASC")
maaiveld = rioxarray.open_rasterio(path_maaiveld)

dir_input = Path(main_dir, "WRIJ_RR_Unpaved_methode_02_input")
dir_input_basis_data = dir_input / "basisdata"

project_areas_path = dir_input_basis_data / "gebieden.gpkg"
path_watergang = dir_input_basis_data / "watergang.gpkg"
path_afwateringseenheden = dir_input_basis_data / "afwateringseenheden.gpkg"
path_hoog_middel_laag = dir_input_basis_data / "hoog_middel_laag.tif"

project_areas = gpd.read_file(project_areas_path, layer="gebieden")
watergang = gpd.read_file(path_watergang)
afwateringseenheden = gpd.read_file(path_afwateringseenheden)
hoog_middel_laag = rioxarray.open_rasterio(path_hoog_middel_laag)

In [23]:
dir_meetdata = Path(main_dir, "WRIJ_RR_Unpaved_methode_01_data\\20260520_meetdata_RR")

bro_peilbuizen

In [24]:
dir_bro_peilbuizen = Path(dir_meetdata, "bro_peilbuizen")
path_bro_peilbuizen = Path(dir_bro_peilbuizen, "alle_gebieden_metadata_min3jaar.gpkg")
bro_peilbuizen = gpd.read_file(path_bro_peilbuizen)
bro_peilbuizen = bro_peilbuizen[bro_peilbuizen["tube_number"]==1]
bro_peilbuizen = bro_peilbuizen[["gmw_bro_id", "geometry"]].rename(columns={"gmw_bro_id": "naam"})

path_bro_peilbuizen_gebied_1 = Path(dir_bro_peilbuizen, "gebied_bergerslag", "gebied_bergerslag_alles.csv")
path_bro_peilbuizen_gebied_2 = Path(dir_bro_peilbuizen, "gebied_pelgrim", "gebied_pelgrim_alles.csv")
path_bro_peilbuizen_gebied_3 = Path(dir_bro_peilbuizen, "gebied_watermolen", "gebied_watermolen_alles.csv")

bro_peilbuizen_gebied_1 = pd.read_csv(path_bro_peilbuizen_gebied_1, parse_dates=["datetime"]).set_index("datetime")
bro_peilbuizen_gebied_2 = pd.read_csv(path_bro_peilbuizen_gebied_2, parse_dates=["datetime"]).set_index("datetime")
bro_peilbuizen_gebied_3 = pd.read_csv(path_bro_peilbuizen_gebied_3, parse_dates=["datetime"]).set_index("datetime")

bro_peilbuizen_metingen = pd.concat([bro_peilbuizen_gebied_1, bro_peilbuizen_gebied_2, bro_peilbuizen_gebied_3])
bro_peilbuizen_metingen = bro_peilbuizen_metingen[bro_peilbuizen_metingen["tube_number"]==1]
bro_peilbuizen_metingen = bro_peilbuizen_metingen.pivot(columns='gmw_bro_id', values='value')

wrij_peilbuizen

In [25]:
dir_wrij_peilbuizen = Path(dir_meetdata, "wrij_peilbuizen_korte_naam")

wrij_peilbuizen = gpd.GeoDataFrame()

# afvoermetingen
csv_files = list(dir_wrij_peilbuizen.glob("*filter1_mNAP.csv"))

wrij_peilbuizen_metingen = pd.DataFrame()
for csv_file in csv_files:
    logging.info(f"Data van: {csv_file.stem}")
    meta, point, df = inlezen_csv_met_metadata(csv_file)
    wrij_peilbuizen = pd.concat([
        wrij_peilbuizen, 
        gpd.GeoDataFrame(
            [{"naam": csv_file.stem.replace("_gws_filter1_mNAP", ""), "geometry": point}],
            geometry="geometry",
            crs=28992
        )
    ])
    wrij_peilbuizen_metingen = pd.merge(
        wrij_peilbuizen_metingen, 
        df[[csv_file.stem]].rename(columns={csv_file.stem: csv_file.stem.replace("_gws_filter1_mNAP", "")}), 
        how="outer", 
        left_index=True, 
        right_index=True
    )

wrij_peilbuizen.to_file(Path(dir_wrij_peilbuizen, "wrij_peilbuizen.gpkg"))

combineren peilbuizen

In [26]:
alle_peilbuizen = pd.concat([bro_peilbuizen, wrij_peilbuizen])

alle_peilbuizen["hoog_middel_laag"] = hoog_middel_laag.sel(band=1).sel(
    x=xr.DataArray(alle_peilbuizen.geometry.x, dims="z"),
    y=xr.DataArray(alle_peilbuizen.geometry.y, dims="z"),
    method="nearest"
).to_dataframe(name="hoog_middel_laag")["hoog_middel_laag"]

alle_peilbuizen["maaiveld"] = maaiveld.sel(band=1).sel(
    x=xr.DataArray(alle_peilbuizen.geometry.x, dims="z"),
    y=xr.DataArray(alle_peilbuizen.geometry.y, dims="z"),
    method="nearest"
).to_dataframe(name="maaiveld")["maaiveld"]

alle_peilbuizen_afw_eenheden = alle_peilbuizen[["naam", "hoog_middel_laag", "maaiveld", "geometry"]].sjoin(afwateringseenheden[["GFEIDENT", "geometry"]])
alle_peilbuizen_afw_eenheden.to_file(Path(dir_meetdata, "alle_peilbuizen.gpkg"))

In [27]:
alle_peilbuizen_metingen = pd.concat([bro_peilbuizen_metingen, wrij_peilbuizen_metingen], axis=1)

### Selecteer welke modelresultaten (gebieden, scenario’s en periode) worden geanalyseerd

In [28]:
# path to the package containing the data
dir_model_basis = Path(main_dir, "WRIJ_RR_Unpaved_methode_03_modellen\\")

# gebied = 0 # Oude IJssel
# gebied = 1 # West
# gebied = 2 # Centraal
# gebied = 3 # Oost

gebieden = {
    1: "gebied_pelgrim", 
    2: "gebied_bergerslag", 
    3: "gebied_watermolen"
}
scenarios = ["REF"] # , "SCEN"]

base_scenario = "REF"

# runs = {
#     "OY_0_kD5_L4": {"name": "kD(5d) L=4xl", "color": "#0072B2"},                                        # blauw
#     "OY_1_kD20_L4": {"name": "kD(20d) L=4xl", "color": "#D55E00"},                                      # oranje
#     "OY_2_kD20_L2": {"name": "kD(20d) L=2xl", "color": "#009E73"},                                      # groen
#     "OY_3_kD20_L4_W": {"name": "kD(20d) L=4xl Wh", "color": "#56B4E9"},                                 # lichtblauw
#     "OY_4_kD20_L4_W_InfMax": {"name": "kD(20d) L=2xl Wh InfMax", "color": "#F0E442"},                   # geel
#     "OY_5_kD20_L4_W_InfMax_InitGWS": {"name": "kD(20d) L=2xl Wh InfMax InitGWS", "color": "#E69F00"},   # goud/oranje
# }

# # LONG TEST
# start_date = "2014-07-1"
# end_date = "2016-09-30"
# seizoenen = ["zomer", "winter", "winter", "zomer"]
# date_range = pd.date_range(start_date, end_date, freq="3MS")

# MODELRESULTATEN TBV ONDERZOEK INLOOPTIJD ICM INITIËLE GRONDWATERSTAND

runs = {"OY_5_kD20_L4_W_InfMax_InitGWS_20140101_20160930": {"name" : "1 jaar inlooptijd", "color" : "#0072B2"},                     # blauw
        "OY_5_kD20_L4_W_InfMax_InitGWS_20130101_20160930": {"name" : "2 jaar inlooptijd", "color" : "#D55E00"},                     # oranje
        "OY_5_kD20_L4_W_InfMax_InitGWS_20120401_20160930": {"name" : "2 jaar en 9 maanden inlooptijd", "color" : "#009E73"},        # groen
}

list_start_date = ["2014-01-01", "2013-01-01", "2012-04-01"]
list_end_date = ["2016-09-30", "2016-09-30", "2016-09-30"]
list_seizoenen = [["winter", "zomer", "zomer", "winter"],
                  ["winter", "zomer", "zomer", "winter"],
                  ["zomer", "zomer", "winter", "winter"]]
list_date_range = [pd.date_range(start_date, end_date, freq="3MS") for start_date, end_date in zip(list_start_date, list_end_date)]

dir_resultaten = Path(main_dir, "WRIJ_RR_Unpaved_methode_04_resultaten\\1_6_onderzoek_inlooptijd")

In [11]:
simulations_total = pd.DataFrame()

for i, run_name in enumerate(runs.keys()):

    start_date = list_start_date[i]
    end_date = list_end_date[i]
    seizoenen = list_seizoenen[i]
    date_range = list_date_range[i]

    for gebied in gebieden:
        for scenario in scenarios:

            simulaties = pd.DataFrame()
            simulaties["start_date"] = date_range
            simulaties["end_date"] = simulaties["start_date"].shift(-1)
            simulaties.loc[simulaties.index[-1],"end_date"] = pd.to_datetime(end_date)
            simulaties["seizoen"] = (seizoenen * 100)[:len(date_range)]
            simulaties["scenario"] = scenario
            simulaties["gebied"] = gebied
            simulaties["restart_in"] = [0] + [1] * len(simulaties.index[1:])
            simulaties["run_name"] = run_name

            simulaties["model_name"] = simulaties.apply(lambda x: f"{run_name[:4]}_gebied{gebied}_{scenario}_rr_{pd.to_datetime(x.start_date).strftime('%Y%m%d')}_{pd.to_datetime(x.end_date).strftime('%Y%m%d')}", axis=1)
            simulations_total = pd.concat([simulations_total, simulaties])

In [14]:
simulations_total.iloc[0:40]

,start_date,end_date,seizoen,scenario,gebied,restart_in,run_name,model_name
0,2014-01-01,2014-04-01,winter,REF,1,0,OY_5_kD20_L4_W_InfMax_InitGWS_20140101_20160930,OY_5_gebied1_REF_rr_20140101_20140401
1,2014-04-01,2014-07-01,zomer,REF,1,1,OY_5_kD20_L4_W_InfMax_InitGWS_20140101_20160930,OY_5_gebied1_REF_rr_20140401_20140701
2,2014-07-01,2014-10-01,zomer,REF,1,1,OY_5_kD20_L4_W_InfMax_InitGWS_20140101_20160930,OY_5_gebied1_REF_rr_20140701_20141001
3,2014-10-01,2015-01-01,winter,REF,1,1,OY_5_kD20_L4_W_InfMax_InitGWS_20140101_20160930,OY_5_gebied1_REF_rr_20141001_20150101
4,2015-01-01,2015-04-01,winter,REF,1,1,OY_5_kD20_L4_W_InfMax_InitGWS_20140101_20160930,OY_5_gebied1_REF_rr_20150101_20150401
5,2015-04-01,2015-07-01,zomer,REF,1,1,OY_5_kD20_L4_W_InfMax_InitGWS_20140101_20160930,OY_5_gebied1_REF_rr_20150401_20150701
6,2015-07-01,2015-10-01,zomer,REF,1,1,OY_5_kD20_L4_W_InfMax_InitGWS_20140101_20160930,OY_5_gebied1_REF_rr_20150701_20151001
7,2015-10-01,2016-01-01,winter,REF,1,1,OY_5_kD20_L4_W_InfMax_InitGWS_20140101_20160930,OY_5_gebied1_REF_rr_20151001_20160101
8,2016-01-01,2016-04-01,winter,REF,1,1,OY_5_kD20_L4_W_InfMax_InitGWS_20140101_20160930,OY_5_gebied1_REF_rr_20160101_20160401
9,2016-04-01,2016-07-01,zomer,REF,1,1,OY_5_kD20_L4_W_InfMax_InitGWS_20140101_20160930,OY_5_gebied1_REF_rr_20160401_20160701


INLEZEN ALLE RUNS, GEBIEDEN, SCENARIOS

In [15]:
total_rr_gwl = dict()

for run, run_dict in runs.items():
    print(f"Run: {run}")
    for gebied in gebieden:
        print(f" * Gebied: {gebied}")
        for scenario in scenarios:
            print(f"   - Scenario: {scenario}")
            simulaties = simulations_total[(simulations_total["run_name"]==run) & (simulations_total["gebied"]==gebied) & (simulations_total["scenario"]==scenario)]
            rr_gwl = pd.DataFrame()
            for index, simulatie in simulaties.iterrows():
                # print(run  + " - " + str(simulatie.gebied) + " - " + simulatie.scenario + " - " + simulatie.model_name)

                dir_model = Path(dir_model_basis, run, f"gebied_{simulatie.gebied}", simulatie.scenario)
                unpaved_rr_file = "upflowdt.his"

                path_unpaved_rr_file = Path(dir_model, simulatie.model_name, "rr", unpaved_rr_file)
                if not path_unpaved_rr_file.exists():
                    print(f"File {path_unpaved_rr_file} does not exist. Skipping.")
                    continue
                rr_his = hkvsobekpy.read_his.ReadMetadata(path_unpaved_rr_file)
                rr_results = rr_his.DataFrame()['Groundw.Level   [m] ']
                rr_gwl = pd.concat([rr_gwl, rr_results])
                total_rr_gwl[f"{run}_{gebied}_{scenario}"] = rr_gwl

Run: OY_5_kD20_L4_W_InfMax_InitGWS_20140101_20160930
 * Gebied: 1
   - Scenario: REF
 * Gebied: 2
   - Scenario: REF
 * Gebied: 3
   - Scenario: REF
Run: OY_5_kD20_L4_W_InfMax_InitGWS_20130101_20160930
 * Gebied: 1
   - Scenario: REF
 * Gebied: 2
   - Scenario: REF
 * Gebied: 3
   - Scenario: REF
Run: OY_5_kD20_L4_W_InfMax_InitGWS_20120401_20160930
 * Gebied: 1
   - Scenario: REF
 * Gebied: 2
   - Scenario: REF
 * Gebied: 3
   - Scenario: REF


### Plot het resultaat

TODO
- gemeten afvoeren gebied 3                                                     --> KLAAR: twijfels bij afvoer gebied 3
- initiële waterstanden                                                         --> KLAAR: geimplementeerd
- L=2x kleine l ipv L = 4x kleine l                                             --> KLAAR: conclusie: L=4xl klopt wel
- grafieken grondwaterstanden van alle meetpunten plus kaartje                  --> KLAAR: zie hieronder
- dynamische grafieken                                                          --> KLAAR: zie hieronder

In [29]:
from pathlib import Path
from uuid import uuid4
import plotly.io as pio

def save_plotly_tabs_html(figures, titles, output_path="plotly_tabs.html"):
    """
    Save multiple Plotly figures into one self-contained HTML file with tabs.

    Parameters
    ----------
    figures : list
        List of Plotly Figure objects.
    titles : list of str
        Tab titles, same length as figures.
    output_path : str or Path
        Output HTML file path.
    """
    if len(figures) != len(titles):
        raise ValueError("figures and titles must have the same length")

    output_path = Path(output_path)

    # Unique IDs for tabs and panels
    tab_ids = [f"tab-{uuid4().hex}" for _ in figures]
    panel_ids = [f"panel-{uuid4().hex}" for _ in figures]

    # Plotly JS included only once
    plotly_js = pio.to_html(
        figures[0],
        full_html=False,
        include_plotlyjs="cdn"
    ).split('<div id="')[0]

    # Generate figure divs without Plotly JS
    figure_divs = []
    for fig, panel_id in zip(figures, panel_ids):
        fig_html = pio.to_html(
            fig,
            full_html=False,
            include_plotlyjs=False,
            div_id=panel_id
        )
        figure_divs.append(fig_html)

    # CSS + JS for tabs
    css = """
    <style>
      body {
        font-family: Arial, sans-serif;
        margin: 0;
        padding: 16px;
      }
      .tab-bar {
        display: flex;
        gap: 8px;
        border-bottom: 1px solid #ddd;
        margin-bottom: 16px;
        flex-wrap: wrap;
      }
      .tab-button {
        appearance: none;
        border: 1px solid #ccc;
        border-bottom: none;
        background: #f7f7f7;
        padding: 10px 14px;
        cursor: pointer;
        border-radius: 8px 8px 0 0;
        font: inherit;
      }
      .tab-button.active {
        background: white;
        border-color: #999;
        font-weight: 600;
      }
      .tab-panel {
        display: none;
      }
      .tab-panel.active {
        display: block;
      }
    </style>
    """

    js = f"""
    <script>
      function showTab(idx) {{
        const buttons = document.querySelectorAll('.tab-button');
        const panels = document.querySelectorAll('.tab-panel');

        buttons.forEach(btn => btn.classList.remove('active'));
        panels.forEach(p => p.classList.remove('active'));

        buttons[idx].classList.add('active');
        panels[idx].classList.add('active');

        // Resize the Plotly chart after becoming visible
        const plotDiv = panels[idx].querySelector('.plotly-graph-div');
        if (plotDiv && window.Plotly) {{
          window.Plotly.Plots.resize(plotDiv);
        }}
      }}

      document.addEventListener('DOMContentLoaded', () => {{
        showTab(0);
      }});
    </script>
    """

    # Build tab buttons and panels
    tab_buttons = "\n".join(
        f'<button class="tab-button" onclick="showTab({i})">{title}</button>'
        for i, title in enumerate(titles)
    )

    tab_panels = "\n".join(
        f'<div class="tab-panel" id="{panel_ids[i]}-wrap">{figure_divs[i]}</div>'
        for i in range(len(figures))
    )

    html = f"""<!doctype html>
<html>
<head>
  <meta charset="utf-8">
  <meta name="viewport" content="width=device-width, initial-scale=1">
  {css}
  {plotly_js}
</head>
<body>
  <div class="tab-bar">
    {tab_buttons}
  </div>
  {tab_panels}
  {js}
</body>
</html>
"""

    output_path.write_text(html, encoding="utf-8")
    return output_path

In [30]:
# dir_data = Path("..\\..\\WRIJ_RR_Unpaved_methode_02_input")

# df_unpaved_zomers = pd.DataFrame()
# df_unpaved_winters = pd.DataFrame()
# df_ernst_zomers = pd.DataFrame()
# df_ernst_winters = pd.DataFrame()

# for run in runs.keys():
#     for gebied in [1,2,3]:
#         dir_gebied = Path(dir_data, "rr_input_area_scenario", run, f"gebied_{gebied}")
        
#         gebied = gpd.read_file(dir_gebied / f"gebied.gpkg", layer=f"gebied")
#         watergang = gpd.read_file(dir_gebied / f"watergang.gpkg", layer=f"watergang")
#         afwateringseenheden = gpd.read_file(dir_gebied / f"afwateringseenheden.gpkg", layer=f"afwateringseenheden")

#         #Read modelinput
#         df_unpaved_zomer = pd.read_csv(dir_gebied / scenario / f"df_unpaved_zomer.csv")
#         df_unpaved_winter = pd.read_csv(dir_gebied / scenario / f"df_unpaved_winter.csv")
#         df_ernst_zomer = pd.read_csv(dir_gebied / scenario / f"df_ernst_zomer.csv")
#         df_ernst_winter = pd.read_csv(dir_gebied / scenario / f"df_ernst_winter.csv")

#         df_unpaved_winters = pd.concat([df_unpaved_winters, df_unpaved_winter])
#         df_unpaved_zomers = pd.concat([df_unpaved_zomers, df_unpaved_zomer])
#         df_ernst_winters = pd.concat([df_ernst_winters, df_ernst_winter])
#         df_ernst_zomers = pd.concat([df_ernst_zomers, df_ernst_zomer])

#         df_unpaved_winters["GFEIDENT"] = df_unpaved_winters.apply(lambda x: x["code"].split("_")[0], axis=1)
#         df_unpaved_zomers["GFEIDENT"] = df_unpaved_zomers.apply(lambda x: x["code"].split("_")[0], axis=1)

#         df_ernst_winters["GFEIDENT"] = df_ernst_winters.apply(lambda x: x["code"].split("_")[0], axis=1)
#         df_ernst_zomers["GFEIDENT"] = df_ernst_zomers.apply(lambda x: x["code"].split("_")[0], axis=1)

#         df_unpaved_winters = df_unpaved_winters[["GFEIDENT", "surface_level", "boundary_waterlevel", "code"]]
#         df_unpaved_zomers = df_unpaved_zomers[["GFEIDENT", "surface_level", "boundary_waterlevel", "code"]]

#         df_ernst_zomers = df_ernst_zomers[["GFEIDENT", "lv", "code"]]
#         df_ernst_winters = df_ernst_winters[["GFEIDENT", "lv", "code"]]

In [32]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# path to the package containing the data
dir_model_basis = Path(main_dir, "WRIJ_RR_Unpaved_methode_03_modellen\\oude_ijssel")

start_plot = "2015-01-01"
end_plot = "2016-09-30"

start_date = start_plot
end_date = end_plot

selectie_gebieden = {
    1: {"ymax": 10}, 
    2: {"ymax": 2},
    3: {"ymax": 10},
}

selectie_runs = runs.copy()

selectie_scenario = {
    "REF": {"linestyle": "solid", "visible": True}}#, 
#     "SCEN": {"linestyle": "dash", "visible": False},
# }
include_metingen = True

metingen_instroom = True
metingen = True
figures = []
figures_titles = []

for i, peilbuis in alle_peilbuizen_afw_eenheden.iterrows():
    fig = make_subplots(
        rows=3, cols=1,
        shared_xaxes=True,
        vertical_spacing=0.03,
        subplot_titles=("RR-knoop - Hoog", "RR-knoop - Middel", "RR-knoop - Laag")
    )

    peilbuis_naam = peilbuis["naam"]
    afw_eenheid = peilbuis["GFEIDENT"]
    print(peilbuis_naam, afw_eenheid)

    # def add_levels_to_figure(start_date, end_date, code, df_unpaved_winters, df_unpaved_zomers, df_ernst_winters, df_ernst_zomers):
    #     if code not in df_unpaved_winters["code"]:
    #         return

    #     df_unpaved_winter = df_unpaved_winters[df_unpaved_winters["code"]==code].iloc[0]
    #     df_unpaved_zomer = df_unpaved_zomers[df_unpaved_zomers["code"]==f"{afw_eenheid}_{level}"].iloc[0]
    #     df_ernst_winter = df_ernst_winters[df_ernst_winters["code"]==f"{afw_eenheid}_{level}"].iloc[0]
    #     df_ernst_zomer = df_ernst_zomer[df_ernst_zomer["code"]==f"{afw_eenheid}_{level}"].iloc[0]

    #     def add_seasonal_levels(years, winterlevel, summerlevel, color="blue", dash="dash"):
    #         for year in years:
    #             # Winter: 1/10 until 1/4
    #             fig.add_shape(
    #                 type="line",
    #                 x0=f"{year}-10-01",
    #                 x1=f"{year+1}-04-01",
    #                 y0=winterlevel,
    #                 y1=winterlevel,
    #                 line=dict(color=color, dash=dash),
    #                 row=i+1, col=1
    #             )

    #             # Summer: 1/4 until 1/10
    #             fig.add_shape(
    #                 type="line",
    #                 x0=f"{year}-04-01",
    #                 x1=f"{year}-10-01",
    #                 y0=summerlevel,
    #                 y1=summerlevel,
    #                 line=dict(color=color, dash=dash),
    #                 row=i+1, col=1
    #             )
    #     years = np.arange(int(start_date[:3])-1, int(end_date[:3])+1)
    #     add_seasonal_levels(years=years, winterlevel=df_unpaved_winter["boundary_waterlevel"], summerlevel=df_unpaved_zomer["boundary_waterlevel"], color="blue", dash="dash")

    peilbuis_metingen = alle_peilbuizen_metingen.loc[start_date:end_date, peilbuis_naam]
    for i, level in enumerate(["hoog", "middel", "laag"]):
        value = 3-i
        # add_levels_to_figure(
        #     start_date=start_date, 
        #     end_date=end_date,
        #     code=f"{afw_eenheid}_{level}",
        #     df_unpaved_winters=df_unpaved_winters,
        #     df_unpaved_zomers=df_unpaved_zomers,
        #     df_ernst_winters=df_ernst_zomers,
        #     df_ernst_zomers=df_ernst_zomers
        # )
        if peilbuis["hoog_middel_laag"] == value:
            fig.add_trace(
                go.Scatter(
                    x=peilbuis_metingen.index, 
                    y=peilbuis_metingen, 
                    mode="markers", 
                    name=f"Gemeten grondwaterstand peilbuis ({level})",
                    marker=dict(color="purple", size=2)
                ),
                row=i+1, col=1
            )
            fig.add_trace(
                go.Scatter(
                    x=[start_date, end_date],
                    y=[peilbuis["maaiveld"], peilbuis["maaiveld"]],
                    mode="lines",
                    name="Maaiveld meetpunt",
                    line=dict(color="purple", dash="dash")
                ),
                row=i+1, col=1
            )

    for scenario, scenario_dict in selectie_scenario.items():
        for run, run_dict in selectie_runs.items():
            for gebied in selectie_gebieden.keys():
                gw = total_rr_gwl[run + "_" + str(gebied) + "_" + scenario]
                cols_bro_peilbuis = [col for col in gw.columns if afw_eenheid in col]
                if cols_bro_peilbuis:
                    break
            legend_visible = True
            for i, level in enumerate(["hoog", "middel", "laag"]):
                col_level = f"unp_{afw_eenheid}_{level}"
                if col_level in gw:
                    gmw_bro_peilbuis = gw[col_level]
                    fig.add_trace(
                        go.Scatter(
                            x=gmw_bro_peilbuis.index, 
                            y=gmw_bro_peilbuis, 
                            mode="lines", 
                            legendgroup=f"{run}_{scenario}",
                            name=f"{scenario} = {run_dict['name']}",
                            showlegend=True if legend_visible else False,
                            visible=True if scenario=="REF" else "legendonly",
                            line=dict(dash=scenario_dict["linestyle"], color=run_dict["color"])
                        ),
                        row=i+1, col=1
                    )
                    legend_visible = False

    fig.update_layout(
        template="simple_white",
        title={
            "text": f"<b>RR-modellering Oude IJssel - Grondwaterstand {peilbuis_naam} - afwateringseenheid {afw_eenheid}</b>",
            "x": 0.05,
            "xanchor": "left"
        },
        margin=dict(l=20, r=20, t=50, b=20),
        height=800,
    )

    fig.update_xaxes(showgrid=True, gridcolor="lightgray", range=[start_plot, end_plot])
    fig.update_yaxes(showgrid=True, gridcolor="lightgray")

    # fig.write_html(Path(dir_model_basis, f"grondwaterstand_{peilbuis_naam}.html"), include_plotlyjs="cdn")
    figures.append(fig)
    figures_titles.append(peilbuis_naam.replace("GMW000000", "GMW"))

save_plotly_tabs_html(figures, figures_titles, output_path=Path(dir_model_basis, Path(dir_resultaten, f"grondwaterstand_pilotgebieden_oude_ijssel.html")));

GMW000000079352 AE54830027
GMW000000079382 AE54840060
GMW000000079384 AE54840072
GMW000000104270 AE54870073
GMW000000104271 AE54870072
GMW000000105884 AE54870073
GMW000000105935 AE54870077
GMW000000106098 AE54870077
GMW000000106139 AE54870073
GMW000000106263 AE54870073
GMW000000106312 AE54870076
GMW000000079354 AE54490003
GMW000000074710 AE54620107
GMW000000074716 AE54620107
GMW000000074718 AE54620107
GMW000000074719 AE54620107
GMW000000074720 AE54620107
GMW000000079129 AE54620130
GMW000000079131 AE54640018
GMW000000079314 AE54620122
GMW000000079322 AE54610010
GMW000000079333 AE54620130
GMW000000079343 AE54620082
GMW000000079344 AE54640005
Aalten_Gendringseweg AE54460017
Azewijn_Hartjensstraat AE54870031
